In [31]:
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # 数值稳定
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def softmax_derivative(z):
    s = softmax(z)
    return s * (1 - s)

# 加载MNIST数据集
def load_mnist():
    with open('mnist.npz', 'rb') as f:
        data = np.load(f)
        train_x, train_y = data['x_train'], data['y_train']
        test_x, test_y = data['x_test'], data['y_test']
    return train_x, train_y, test_x, test_y

# 对卷积层的参数，其本身形如conv1['weights'].value
# 同时具有梯度grad，可通过如 
# conv1['weights'].grad = your_calculation
# 对梯度进行保存
class nn_Parameter:
    def __init__(self, value):
        self.value = value
        self.grad = np.zeros_like(value)

# CNN网络结构类
class SimpleCNN:
    def __init__(self):
        
        self.conv1 = {
            'weights': nn_Parameter(np.random.randn(5, 5, 1, 6)),
            'bias': nn_Parameter(np.zeros(6)),
            'output': np.zeros((24, 24, 6))
        }

        self.pool1 = {
            'weights': np.ones((2, 2))/4,
            'output': np.zeros((12, 12, 6))
        }

        self.conv2 = {
            'weights': nn_Parameter(np.random.randn(5, 5, 6, 12)),
            'bias': nn_Parameter(np.zeros(12)),
            'output': np.zeros((8, 8, 12))
        }

        self.pool2 = {
            'weights': np.ones((2, 2))/4,
            'output': np.zeros((4, 4, 12))
        }

        self.fc = {
            'weights': nn_Parameter(np.random.randn(12*4*4,10)),
            'bias': nn_Parameter(np.zeros(10)),
            'output': np.zeros((10))
        }

    @staticmethod
    def conv2d(input, filters, bias, stride=1, padding=0):
        # input   : H x W x C_in x N; N是批次大小
        # filters : F_h x F_w x C_in x c_out; c_out 是每一个卷积核产生的输出通道数
        # bias    : c_out
        assert input.shape[2] == filters.shape[2], "Input channels must match filter channels"

        if padding > 0:
            input = np.pad(input, ((padding, padding), (padding, padding), (0, 0)), mode='constant')

        ch_in = filters.shape[2]
        ch_out = filters.shape[3]
        ch_width = (input.shape[0] - filters.shape[0]) // stride + 1
        ch_height = (input.shape[1] - filters.shape[1]) // stride + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_out, N))

        in_h, in_w = input.shape[0], input.shape[1]
        k_h, k_w = filters.shape[0], filters.shape[1]
        # 按输入通道遍历
        for n in range(N):
            for c in range(ch_in):
                # 按输出通道遍历，计算输入对应输出通道的贡献
                for k in range(ch_out):
                    _region = input[:, :, c, n]
                    _kernel = filters[:, :, c, k]
                    for i in range(0, in_h - k_h + 1, stride):
                        for j in range(0, in_w - k_w + 1, stride):
                            res[i//stride, j//stride, k, n] = np.sum(
                                _region[i:i+k_h, j:j+k_w] * _kernel
                            )
                    res[:, :, k, n] += bias[k]
        return res

    @staticmethod
    def avg_pool2d(input, kernel_size=2):
        # input : H x W x C x N; N是批次大小
        # 默认丢弃边界不完整部分
        ch_in = input.shape[2]
        ch_width = (input.shape[0] - kernel_size) // kernel_size + 1
        ch_height = (input.shape[1] - kernel_size) // kernel_size + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_in, N))

        in_h, in_w = input.shape[0], input.shape[1]
        for n in range(N):
            for c in range(ch_in):
                _region = input[:, :, c, n]
                for i in range(0, in_h - kernel_size + 1, kernel_size):
                    for j in range(0, in_w - kernel_size + 1, kernel_size):
                        res[i//kernel_size, j//kernel_size, c, n] = np.mean(
                            _region[i:i+kernel_size, j:j+kernel_size]
                        )
        return res

    @staticmethod
    def max_pool2d(input, kernel_size=2):
        # input : H x W x C x N; N是批次大小
        # 默认丢弃边界不完整部分
        ch_in = input.shape[2]
        ch_width = (input.shape[0] - kernel_size) // kernel_size + 1
        ch_height = (input.shape[1] - kernel_size) // kernel_size + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_in, N))

        in_h, in_w = input.shape[0], input.shape[1]
        for n in range(N):
            for c in range(ch_in):
                _region = input[:, :, c, n]
                for i in range(0, in_h - kernel_size + 1, kernel_size):
                    for j in range(0, in_w - kernel_size + 1, kernel_size):
                        res[i//kernel_size, j//kernel_size, c, n] = np.max(
                            _region[i:i+kernel_size, j:j+kernel_size]
                        )
        return res

    @staticmethod
    def relu(x):
        return np.maximum(0, x)

    @staticmethod
    def relu_derivative(x):
        return (x > 0).astype(float)
    # 前向传播函数 请编写其中的内容
    def forward(self, x):
        # 输入数据是 N x H x W 的形式，N是批次大小，H和W是图像的高度和宽度
        # 卷积层和池化层的输入输出都是 H x W x C x N 的形式，C是通道数
        # 对输入数据进行维度调整
        inputs = np.transpose(x, (1, 2, 0))[:, :, np.newaxis, :]

        # conv1 -> relu1 -> pool1 -> conv2 -> relu2 -> pool2 -> fc -> softmax
        output = self.relu(self.conv2d(inputs, self.conv1['weights'].value, self.conv1['bias'].value))
        self.conv1['output'] = output

        output = self.avg_pool2d(output, kernel_size=2)
        self.pool1['output'] = output

        output = self.relu(self.conv2d(output, self.conv2['weights'].value, self.conv2['bias'].value))
        self.conv2['output'] = output

        output = self.avg_pool2d(output, kernel_size=2)
        self.pool2['output'] = output


        # 调整输出，按照batch维度展平
        output = output.reshape(output.shape[3], -1)  # N x (12*4*4)
        output = output @ self.fc['weights'].value + self.fc['bias'].value
        output = softmax(output)

        self.fc['output'] = output
        

    # 后向传播函数 请编写其中的内容
    def backward(self, y_true):
        # y_true : N x 1
        # fc_output : N x 10
        # 反向传播
        fc_output = self.fc['output']
        N = y_true.shape[0]

        loss = -np.mean(
            np.log(
                fc_output[np.arange(N), y_true.flatten()] + 1e-12
            )
        )

        d = fc_output.copy()
        d[np.arange(N), y_true.flatten()] -= 1
        d /= N

        input = self.conv2["output"].value.reshape(N, -1)  # N x (12*4*4)
        self.fc['weight'].grad  = np.dot(input.T, d)
        self.fc['bias'].grad = np.sum(d, axis=0)

        d = d @ self.fc['weights'].value.T  # N x (12*4*4)
        input = self.pool2["output"].value.reshape(N, -1)  # N x (12*4*4)
        self.conv2['weight'].grad =
        self.conv2['bias'].grad = 0
        
        self.conv1['weight'].grad = 0
        self.conv1['bias'].grad = 0

    # 更新权重函数 请编写其中的内容
    def update(self, learning_rate):
        self.fc['weight'].value += self.fc['weight'].grad
        self.fc['bias'].value += self.fc['bias'].grad
        self.conv2['weight'].value += self.conv2['weight'].grad
        self.conv2['bias'].value += self.conv2['bias'].grad
        self.conv1['weight'].value += self.conv1['weight'].grad
        self.conv1['bias'].value += self.conv1['bias'].grad








In [31]:
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # 数值稳定
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def softmax_derivative(z):
    s = softmax(z)
    return s * (1 - s)

# 加载MNIST数据集
def load_mnist():
    with open('mnist.npz', 'rb') as f:
        data = np.load(f)
        train_x, train_y = data['x_train'], data['y_train']
        test_x, test_y = data['x_test'], data['y_test']
    return train_x, train_y, test_x, test_y

# 对卷积层的参数，其本身形如conv1['weights'].value
# 同时具有梯度grad，可通过如
# conv1['weights'].grad = your_calculation
# 对梯度进行保存
class nn_Parameter:
    def __init__(self, value):
        self.value = value
        self.grad = np.zeros_like(value)

# CNN网络结构类
class SimpleCNN:
    def __init__(self):

        self.conv1 = {
            'weights': nn_Parameter(np.random.randn(5, 5, 1, 6)),
            'bias': nn_Parameter(np.zeros(6)),
            'output': np.zeros((24, 24, 6))
        }

        self.pool1 = {
            'weights': np.ones((2, 2))/4,
            'output': np.zeros((12, 12, 6))
        }

        self.conv2 = {
            'weights': nn_Parameter(np.random.randn(5, 5, 6, 12)),
            'bias': nn_Parameter(np.zeros(12)),
            'output': np.zeros((8, 8, 12))
        }

        self.pool2 = {
            'weights': np.ones((2, 2))/4,
            'output': np.zeros((4, 4, 12))
        }

        self.fc = {
            'weights': nn_Parameter(np.random.randn(12*4*4,10)),
            'bias': nn_Parameter(np.zeros(10)),
            'output': np.zeros((10))
        }

    @staticmethod
    def conv2d(input, filters, bias, stride=1, padding=0):
        # input   : H x W x C_in x N; N是批次大小
        # filters : F_h x F_w x C_in x c_out; c_out 是每一个卷积核产生的输出通道数
        # bias    : c_out
        assert input.shape[2] == filters.shape[2], "Input channels must match filter channels"

        if padding > 0:
            input = np.pad(input, ((padding, padding), (padding, padding), (0, 0)), mode='constant')

        ch_in = filters.shape[2]
        ch_out = filters.shape[3]
        ch_width = (input.shape[0] - filters.shape[0]) // stride + 1
        ch_height = (input.shape[1] - filters.shape[1]) // stride + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_out, N))

        in_h, in_w = input.shape[0], input.shape[1]
        k_h, k_w = filters.shape[0], filters.shape[1]
        # 按输入通道遍历
        for n in range(N):
            for c in range(ch_in):
                # 按输出通道遍历，计算输入对应输出通道的贡献
                for k in range(ch_out):
                    _region = input[:, :, c, n]
                    _kernel = filters[:, :, c, k]
                    for i in range(0, in_h - k_h + 1, stride):
                        for j in range(0, in_w - k_w + 1, stride):
                            res[i//stride, j//stride, k, n] = np.sum(
                                _region[i:i+k_h, j:j+k_w] * _kernel
                            )
                    res[:, :, k, n] += bias[k]
        return res

    @staticmethod
    def avg_pool2d(input, kernel_size=2):
        # input : H x W x C x N; N是批次大小
        # 默认丢弃边界不完整部分
        ch_in = input.shape[2]
        ch_width = (input.shape[0] - kernel_size) // kernel_size + 1
        ch_height = (input.shape[1] - kernel_size) // kernel_size + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_in, N))

        in_h, in_w = input.shape[0], input.shape[1]
        for n in range(N):
            for c in range(ch_in):
                _region = input[:, :, c, n]
                for i in range(0, in_h - kernel_size + 1, kernel_size):
                    for j in range(0, in_w - kernel_size + 1, kernel_size):
                        res[i//kernel_size, j//kernel_size, c, n] = np.mean(
                            _region[i:i+kernel_size, j:j+kernel_size]
                        )
        return res

    @staticmethod
    def max_pool2d(input, kernel_size=2):
        # input : H x W x C x N; N是批次大小
        # 默认丢弃边界不完整部分
        ch_in = input.shape[2]
        ch_width = (input.shape[0] - kernel_size) // kernel_size + 1
        ch_height = (input.shape[1] - kernel_size) // kernel_size + 1
        N = input.shape[3]

        res = np.zeros((ch_width, ch_height, ch_in, N))

        in_h, in_w = input.shape[0], input.shape[1]
        for n in range(N):
            for c in range(ch_in):
                _region = input[:, :, c, n]
                for i in range(0, in_h - kernel_size + 1, kernel_size):
                    for j in range(0, in_w - kernel_size + 1, kernel_size):
                        res[i//kernel_size, j//kernel_size, c, n] = np.max(
                            _region[i:i+kernel_size, j:j+kernel_size]
                        )
        return res

    @staticmethod
    def relu(x):
        return np.maximum(0, x)


    @staticmethod
    def conv2d_backward(x, w, dY):
        """
        x : (N, Cin, H, W)
        w : (Cout, Cin, Kh, Kw)
        dY: (N, Cout, Hout, Wout)
        """
        N, Cin, H, W = x.shape
        Cout, _, Kh, Kw = w.shape
        Hout, Wout = dY.shape[2], dY.shape[3]

        # bias gradient
        db = np.sum(dY, axis=(0,2,3))

        # weight gradient
        dW = np.zeros_like(w)

        for n in range(N):
            for c_out in range(Cout):
                for i in range(Hout):
                    for j in range(Wout):
                        patch = x[n, :, i:i+Kh, j:j+Kw]
                        dW[c_out] += patch * dY[n, c_out, i, j]

        # input gradient
        dx = np.zeros_like(x)

        w_rot = w[:, :, ::-1, ::-1]  # 180°翻转

        pad_h = Kh - 1
        pad_w = Kw - 1

        dY_pad = np.pad(dY, ((0,0),(0,0),(pad_h,pad_h),(pad_w,pad_w)))

        for n in range(N):
            for c in range(Cin):
                for i in range(H):
                    for j in range(W):
                        region = dY_pad[n, :, i:i+Kh, j:j+Kw]
                        dx[n,c,i,j] = np.sum(region * w_rot[:, c])

        return dx, dW, db

    # 前向传播函数 请编写其中的内容
    def forward(self, x):
        # 输入数据是 N x H x W 的形式，N是批次大小，H和W是图像的高度和宽度
        # 卷积层和池化层的输入输出都是 H x W x C x N 的形式，C是通道数
        # 对输入数据进行维度调整
        inputs = np.transpose(x, (1, 2, 0))[:, :, np.newaxis, :]

        # conv1 -> relu1 -> pool1 -> conv2 -> relu2 -> pool2 -> fc -> softmax
        output = self.relu(self.conv2d(inputs, self.conv1['weights'].value, self.conv1['bias'].value))
        self.conv1['output'] = output

        output = self.avg_pool2d(output, kernel_size=2)
        self.pool1['output'] = output

        output = self.relu(self.conv2d(output, self.conv2['weights'].value, self.conv2['bias'].value))
        self.conv2['output'] = output

        output = self.avg_pool2d(output, kernel_size=2)
        self.pool2['output'] = output

        # 调整输出，按照batch维度展平
        output = output.reshape(output.shape[3], -1)  # N x (12*4*4)
        output = output @ self.fc['weights'].value + self.fc['bias'].value
        output = softmax(output)

        self.fc['output'] = output


    # 后向传播函数 请编写其中的内容
    def backward(self, y_true):
        # y_true : N x 1
        # fc_output : N x 10
        # 反向传播
        fc_output = self.fc['output']
        N = y_true.shape[0]
        loss = -np.mean(
            np.log(
                fc_output[np.arange(N), y_true.flatten()] + 1e-12
            )
        )

        dz = fc_output.copy()
        dz[np.arange(N), y_true.flatten()] -= 1
        dz /= N

        self.fc['weight'].grad  = np.dot(self.conv2["output"].value, dz)
        self.fc['bias'].grad = np.sum(dz, axis=0)

        self.conv2['weight'].grad = 0
        self.conv2['bias'].grad = 0

        self.conv1['weight'].grad = 0
        self.conv1['bias'].grad = 0

    # 更新权重函数 请编写其中的内容
    def update(self, learning_rate):
        self.fc['weight'].value += self.fc['weight'].grad
        self.fc['bias'].value += self.fc['bias'].grad
        self.conv2['weight'].value += self.conv2['weight'].grad
        self.conv2['bias'].value += self.conv2['bias'].grad
        self.conv1['weight'].value += self.conv1['weight'].grad
        self.conv1['bias'].value += self.conv1['bias'].grad








In [ ]:

# 加载MNIST数据集
train_x, train_y, test_x, test_y = load_mnist()

train_x = np.reshape(train_x, (60000, 28, 28)) / 255.0
test_x = np.reshape(test_x, (10000, 28, 28)) / 255.0
train_y = train_y.astype(float)
test_y = test_y.astype(float)



In [ ]:

# 初始化模型
cnn = SimpleCNN()

# 超参数设置
learning_rate = 1
num_epochs = 2
batch_size = 20

total_train = len(train_x)

# 训练模型
for epoch in range(num_epochs):
    for i in range(0, total_train, batch_size):
        # 从训练数据中获取当前批次的数据和标签
        batch_x = train_x[i:i+batch_size]
        batch_y = train_y[i:i+batch_size]

        # 前向传播
        for x, y_true in zip(batch_x, batch_y):
            cnn.forward(x)

        # 后向传播
        for x, y_true in zip(batch_x, batch_y):
            cnn.backward(y_true)

        # 更新权重
        cnn.update(learning_rate)

# 测试模型
predictions = [np.argmax(cnn.forward(x)) for x in test_x]
accuracy = np.mean(np.array(predictions) == test_y)
print(f'Test accuracy: {accuracy * 100}%')
